In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np

# 1. Environment
class GridEnv:
    def __init__(self):
        self.reset()

    def reset(self):
        self.pos = 0
        return self.pos

    def step(self, action):
        if action == 0:       # Left
            self.pos = max(0, self.pos - 1)
        else:                 # Right
            self.pos = min(4, self.pos + 1)

        done = self.pos == 4
        reward = 1.0 if done else -0.01
        return self.pos, reward, done


# 2. Actor Network
class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, state):
        return torch.softmax(self.net(state), dim=-1)


# 3. Critic Network
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, state):
        return self.net(state)


# One-hot state encoding
def encode_state(pos):
    state = torch.zeros(5)
    state[pos] = 1.0
    return state


# 4. Initialize
env = GridEnv()
actor = Actor()
critic = Critic()

actor_optimizer = optim.Adam(actor.parameters(), lr=0.001)
critic_optimizer = optim.Adam(critic.parameters(), lr=0.005)

gamma = 0.99
episodes = 1000
entropy_beta = 0.01


# 5. A2C Training
for episode in range(episodes):
    state = env.reset()

    log_probs = []
    values = []
    rewards = []
    entropies = []

    done = False
    total_reward = 0

    while not done:
        state_tensor = encode_state(state)

        # Actor: action probability
        probs = actor(state_tensor)
        dist = Categorical(probs)
        action = dist.sample()

        # Critic: estimate state value
        value = critic(state_tensor)

        # Environment interaction
        next_state, reward, done = env.step(action.item())

        # Store experience
        log_probs.append(dist.log_prob(action))
        values.append(value.squeeze())
        rewards.append(reward)
        entropies.append(dist.entropy())

        state = next_state
        total_reward += reward

    # 6. Calculate discounted returns
    returns = []
    R = 0.0

    for reward in reversed(rewards):
        R = reward + gamma * R
        returns.insert(0, R)

    returns = torch.tensor(returns, dtype=torch.float32)
    values = torch.stack(values)
    log_probs = torch.stack(log_probs)
    entropies = torch.stack(entropies)

    # 7. Advantage
    advantages = returns - values.detach()

    # 8. Actor loss
    actor_loss = -(log_probs * advantages).mean()
    actor_loss -= entropy_beta * entropies.mean()

    # 9. Critic loss
    critic_loss = nn.functional.mse_loss(values, returns)

    # 10. Update networks
    actor_optimizer.zero_grad()
    actor_loss.backward()
    actor_optimizer.step()

    critic_optimizer.zero_grad()
    critic_loss.backward()
    critic_optimizer.step()

    if (episode + 1) % 100 == 0:
        print(
            f"Episode {episode+1}, "
            f"Reward: {total_reward:.2f}, "
            f"Actor Loss: {actor_loss.item():.4f}, "
            f"Critic Loss: {critic_loss.item():.4f}"
        )


# 11. Test trained Actor
state = env.reset()
done = False
path = [state]

while not done:
    with torch.no_grad():
        probs = actor(encode_state(state))
        action = torch.argmax(probs).item()

    state, reward, done = env.step(action)
    path.append(state)

print("Learned path:", path)
